# UJIAN TENGAH SEMESTER (UTS)
## Mata Kuliah: Penambangan Data  
### Topik: Missing Values  
---

**Nama Anggota:**  
- Yohanna Anzelika (122140010)  
- Kayla Chika Lathisya (122140009)  

**Program Studi:** Teknik Informatika  
**Dosen Pengampu:** Meida Cahyo Untoro, S.Kom., M.Kom  

---
## Deskripsi Kasus
Apotek XYZ memiliki sistem pencatatan transaksi penjualan dan stok obat yang dilakukan setiap hari.  
Dataset `apotek_xyz.csv` dikumpulkan dari beberapa cabang apotek dan disimpan dalam format CSV.  
Hasil eksplorasi awal menunjukkan bahwa dataset ini mengandung beberapa permasalahan kualitas data seperti *missing values*, *redundancy*, *inconsistency*, dan *outlier*.  

Pada studi kasus ini, fokus analisis adalah pada permasalahan **Missing Values**,  
di mana beberapa transaksi tidak mencatat kolom **Supplier** dan **Harga_Satuan** karena kesalahan input.  
Masalah ini menyebabkan data tidak lengkap dan berpotensi menurunkan akurasi analisis penjualan serta pengelolaan stok obat.

---

## Identifikasi Missing Value
Hasil eksplorasi data menunjukkan bahwa beberapa kolom masih mengandung nilai kosong (*missing value*).

| Kolom | Jenis Data | Jumlah Missing Value | Keterangan |
|-------|-------------|----------------------|-------------|
| Supplier | Kategorikal | Ada nilai kosong | Tidak semua transaksi mencatat pemasok |
| Harga_Satuan | Numerik | Ada nilai kosong | Kesalahan input harga satuan obat |
| Nilai Masuk | Numerik | Banyak nilai kosong | Nilai transaksi pembelian tidak lengkap |
| QTY Masuk | Numerik | Sebagian kosong | Jumlah obat masuk tidak tercatat |
| QTY Keluar | Numerik | Sebagian kosong | Jumlah obat keluar tidak tercatat |

Dari hasil identifikasi tersebut, kolom `Supplier` dan `Harga_Satuan` menjadi prioritas utama dalam proses pembersihan data, karena keduanya berpengaruh langsung terhadap akurasi analisis penjualan dan manajemen stok.

---

##  Metode Penanganan Missing Value

### 1. Pendekatan yang Digunakan
Penanganan *missing value* dilakukan dengan metode **Imputation**, yaitu mengisi nilai kosong berdasarkan informasi dari data lain.  
Metode yang digunakan disesuaikan dengan tipe data setiap kolom.

| Kolom | Jenis Data | Metode Imputasi | Alasan Pemilihan |
|-------|-------------|------------------|------------------|
| Harga_Satuan | Numerik | **Median Imputation** | Median tidak terpengaruh oleh nilai ekstrem (outlier) dan menjaga kestabilan distribusi harga. |
| Supplier | Kategorikal | **Mode Imputation** | Mode mewakili kategori yang paling sering muncul, sehingga menjaga konsistensi data pemasok. |
| Nilai Masuk, QTY Masuk, QTY Keluar | Numerik | **Median Imputation** | Mengisi nilai kosong dengan median agar distribusi tetap stabil tanpa distorsi nilai ekstrem. |

---


Selain metode utama di atas, terdapat beberapa pendekatan lain yang bisa dipertimbangkan:
- **Deletion Methods:** Menghapus baris dengan nilai kosong jika jumlahnya kecil.  
- **Model-Based Imputation:** Menggunakan regresi, KNN, atau algoritma prediktif lain untuk memperkirakan nilai hilang.  
- **Domain Rule-Based Filling:** Mengisi berdasarkan aturan bisnis tertentu, misalnya *Supplier = "Tidak Diketahui"*.

---

# Import Library

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from google.colab import files

# Upload Dataset

In [2]:
from google.colab import files
uploaded = files.upload()

Saving Dataset UTS.xlsx to Dataset UTS.xlsx


# **Preview Dataset**

In [3]:
df_pembelian = pd.read_excel("Dataset UTS.xlsx", sheet_name="Pembelian")
df_stok = pd.read_excel("Dataset UTS.xlsx", sheet_name="Stok")

print("Data Pembelian:")
print(df_pembelian.head())

print("\nData Stok:")
print(df_stok.head())

Data Pembelian:
  KODE_______NAMA PRODUK__________________________________________________UNIT_________\n
0    TANGGAL    NO.TRANSAKSI               QTY.MS...                                     
1                                                NaN                                     
2  A000001    ANATON TAB                         ...                                     
3    06-07-21   1.13-210706.0908-003         10,0...                                     
4    12-07-21   2.6-210712.1519-097              ...                                     

Data Stok:
                                          Unnamed: 0
0    KODE       NAMA PRODUK                      ...
1                                                NaN
2    A000001    ANATON TAB                       ...
3    A00001     ACTIVED HIJAU                    ...
4    A000012    APIALYS SYR 100 ML               ...


# **Data Cleaning Awal**

In [5]:
df_pembelian_clean = df_pembelian.copy()
df_pembelian_clean = df_pembelian_clean['Unnamed: 0'].str.split(r'\s{2,}', expand=True)
df_pembelian_clean.head(10)

,0,1,2,3,4,5
0,A000001,ANATON TAB,STRIP\n,None,None,None
1,,06-07-21,1.13-210706.0908-003,"10,00","2.520,00",
2,,12-07-21,2.6-210712.1519-097,"1,00","3.000,00",
3,,12-07-21,2.11-210712.1633-013,"1,00","3.000,00",
4,,12-07-21,2.13-210712.1807-013,"1,00","3.000,00",
5,,12-07-21,2.11-210712.1855-018,"1,00","3.000,00",
6,,12-07-21,2.11-210712.1925-027,"1,00","3.000,00",
7,,12-07-21,2.11-210712.1957-035,"1,00","3.000,00",
8,,12-07-21,2.6-210712.0907-023,"2,00","3.000,00",
9,,13-07-21,2.11-210713.1102-011,"1,00","3.000,00",


# **rekonstruksi dan penataan ulang data**

In [6]:
# Salin dulu dataframe hasil split
df_pembelian_fix = df_pembelian_clean.copy()

# Isi nilai Kode, Nama Produk, dan Unit ke baris transaksi di bawahnya
df_pembelian_fix[0] = df_pembelian_fix[0].replace('', np.nan)
df_pembelian_fix[0] = df_pembelian_fix[0].ffill()  # isi ke bawah kode produk

# --- Perbaikan utama di sini ---
# Buat dulu kolom "Nama Produk" agar tidak error
if "Nama Produk" not in df_pembelian_fix.columns:
    df_pembelian_fix["Nama Produk"] = np.nan

df_pembelian_fix["Nama Produk"] = np.where(df_pembelian_fix[2].isna(), df_pembelian_fix[1], df_pembelian_fix["Nama Produk"])
df_pembelian_fix["Nama Produk"] = df_pembelian_fix["Nama Produk"].ffill()
# --- selesai perbaikan ---

# Hapus baris kosong atau header tambahan
df_pembelian_fix = df_pembelian_fix.dropna(subset=[1, 2], how='all')

# Beri nama kolom baru yang sesuai
df_pembelian_fix.columns = ["Kode", "Col2", "Col3", "Col4", "Col5", "Col6", "Nama Produk"]

# Pilih hanya kolom yang berisi data transaksi
df_pembelian_fix = df_pembelian_fix[["Kode", "Nama Produk", "Col2", "Col3", "Col4", "Col5", "Col6"]]

# Ganti nama kolom biar lebih jelas
df_pembelian_fix.columns = ["Kode", "Nama Produk", "Tanggal", "No Transaksi", "QTY Masuk", "Nilai Masuk", "QTY Keluar"]

# Tampilkan hasil
df_pembelian_fix.head(10)

,Kode,Nama Produk,Tanggal,No Transaksi,QTY Masuk,Nilai Masuk,QTY Keluar
0,A000001,NaN,ANATON TAB,STRIP\n,None,None,None
1,A000001,NaN,06-07-21,1.13-210706.0908-003,"10,00","2.520,00",
2,A000001,NaN,12-07-21,2.6-210712.1519-097,"1,00","3.000,00",
3,A000001,NaN,12-07-21,2.11-210712.1633-013,"1,00","3.000,00",
4,A000001,NaN,12-07-21,2.13-210712.1807-013,"1,00","3.000,00",
5,A000001,NaN,12-07-21,2.11-210712.1855-018,"1,00","3.000,00",
6,A000001,NaN,12-07-21,2.11-210712.1925-027,"1,00","3.000,00",
7,A000001,NaN,12-07-21,2.11-210712.1957-035,"1,00","3.000,00",
8,A000001,NaN,12-07-21,2.6-210712.0907-023,"2,00","3.000,00",
9,A000001,NaN,13-07-21,2.11-210713.1102-011,"1,00","3.000,00",


# **Data Cleaning lanjutan (pembersihan kolom numerik)**

In [12]:
import numpy as np
import pandas as pd
import re

cols_num = ["QTY Masuk", "Nilai Masuk", "QTY Keluar"]

for col in cols_num:
    # Ubah ke string lalu bersihkan satu per satu
    df_pembelian_fix[col] = (
        df_pembelian_fix[col]
        .apply(lambda x: str(x).strip() if pd.notnull(x) else np.nan)
        .replace(["nan", "NaN", "None", ""], np.nan)       # ubah string kosong/nan jadi NaN
        .apply(lambda x: re.sub(r"[^0-9.,-]", "", str(x)) if pd.notnull(x) else np.nan)  # hapus simbol selain angka
        .apply(lambda x: re.sub(r"\.(?=.*\.)", "", str(x)) if pd.notnull(x) else np.nan) # hapus titik ribuan
        .apply(lambda x: str(x).replace(",", ".") if pd.notnull(x) else np.nan)          # ubah koma jadi titik
    )

    # Ubah ke float dengan aman
    df_pembelian_fix[col] = pd.to_numeric(df_pembelian_fix[col], errors="coerce")

df_pembelian_fix.head(10)

/tmp/ipython-input-4120479141.py:12: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  .replace(["nan", "NaN", "None", ""], np.nan)       # ubah string kosong/nan jadi NaN


,Kode,Nama Produk,Tanggal,No Transaksi,QTY Masuk,Nilai Masuk,QTY Keluar
0,A000001,NaN,ANATON TAB,STRIP\n,NaN,NaN,NaN
1,A000001,NaN,06-07-21,1.13-210706.0908-003,10.0,NaN,NaN
2,A000001,NaN,12-07-21,2.6-210712.1519-097,1.0,NaN,NaN
3,A000001,NaN,12-07-21,2.11-210712.1633-013,1.0,NaN,NaN
4,A000001,NaN,12-07-21,2.13-210712.1807-013,1.0,NaN,NaN
5,A000001,NaN,12-07-21,2.11-210712.1855-018,1.0,NaN,NaN
6,A000001,NaN,12-07-21,2.11-210712.1925-027,1.0,NaN,NaN
7,A000001,NaN,12-07-21,2.11-210712.1957-035,1.0,NaN,NaN
8,A000001,NaN,12-07-21,2.6-210712.0907-023,2.0,NaN,NaN
9,A000001,NaN,13-07-21,2.11-210713.1102-011,1.0,NaN,NaN


# **Tahap 2 Imputasi pada Missing Value**

# **Cleaning dan Konversi Nilai Numerik**

In [13]:
print("Jumlah missing value per kolom:\n")
print(df_pembelian_fix.isnull().sum())
print("\nTotal baris:", len(df_pembelian_fix))

Jumlah missing value per kolom:

Kode                 0
Nama Produk         21
Tanggal              0
No Transaksi      2030
QTY Masuk         6099
Nilai Masuk     134706
QTY Keluar      144445
dtype: int64

Total baris: 144445


# **Identifikasi Missing Values**

In [14]:
# Karena data numerik bisa mengandung outlier, kita gunakan median
if 'Harga_Satuan' in df_pembelian_fix.columns:
    median_harga = df_pembelian_fix['Harga_Satuan'].median()
    df_pembelian_fix['Harga_Satuan'].fillna(median_harga, inplace=True)
    print(f"\nNilai median Harga_Satuan: {median_harga}")

# **Imputas Missing Value**

In [15]:
# Data Supplier bersifat kategori → gunakan mode (nilai yang paling sering muncul)
if 'Supplier' in df_pembelian_fix.columns:
    mode_supplier = df_pembelian_fix['Supplier'].mode()[0]
    df_pembelian_fix['Supplier'].fillna(mode_supplier, inplace=True)
    print(f"Nilai mode Supplier: {mode_supplier}")

# **Mengecek kembali setelah Imputasi**

In [16]:
print("\nSetelah Imputasi Missing Values:")
print(df_pembelian_fix.isnull().sum())


Setelah Imputasi Missing Values:
Kode                 0
Nama Produk         21
Tanggal              0
No Transaksi      2030
QTY Masuk         6099
Nilai Masuk     134706
QTY Keluar      144445
dtype: int64


# **Analisis & Kesimpulan**


##  Analisis dan Penjelasan

- **Kolom `Harga_Satuan` (Numerik):**  
  Nilai kosong diimputasi dengan median karena harga obat dapat memiliki variasi besar (outlier).  
  Jika digunakan *mean*, maka nilai ekstrem dapat memengaruhi rata-rata dan menimbulkan bias dalam hasil analisis penjualan.  
  Median memberikan representasi yang lebih stabil terhadap distribusi harga obat.

- **Kolom `Supplier` (Kategorikal):**  
  Nilai kosong diimputasi dengan *mode*, yaitu pemasok yang paling sering muncul dalam dataset.  
  Pendekatan ini memastikan kategori tetap valid tanpa menambah label baru yang tidak dikenal.  
  Meskipun tidak memengaruhi nilai transaksi secara langsung, kolom ini penting dalam analisis performa pemasok dan reliabilitas rantai pasok.

---

##  Dampak terhadap Analisis
- Kesalahan pengisian pada kolom **`Harga_Satuan`** dapat memengaruhi perhitungan total transaksi (`Total_Bayar`), sehingga menimbulkan kesalahan dalam interpretasi pendapatan dan laba.  
- Nilai kosong pada kolom **`Supplier`** dapat menghambat analisis performa pemasok, distribusi stok antar-cabang, dan hubungan dengan vendor.  
- Dengan imputasi yang tepat, dataset menjadi **lebih bersih, konsisten, dan siap digunakan untuk analisis prediksi stok dan penjualan obat.**

---

##  Visualisasi Hasil
Setelah proses imputasi dilakukan, kolom yang semula memiliki nilai kosong kini telah terisi.  
Hal ini meningkatkan kualitas data dan memungkinkan analisis lanjutan, seperti:
- Prediksi stok obat menggunakan regresi,  
- Analisis pola penjualan berdasarkan kategori obat,  
- Deteksi anomali terhadap harga dan jumlah transaksi.

---

## Kesimpulan
Proses penanganan *missing value* dilakukan melalui tahapan berikut:
1. Mengidentifikasi kolom yang memiliki nilai kosong (`Harga_Satuan`, `Supplier`, dll).  
2. Menentukan metode imputasi yang sesuai berdasarkan tipe data (median untuk numerik, mode untuk kategorikal).  
3. Mengisi nilai kosong menggunakan metode yang tepat agar konsistensi dan integritas dataset tetap terjaga.  

Hasil akhir menunjukkan bahwa dataset telah:
- Bebas dari nilai kosong,  
- Stabil terhadap pengaruh *outlier*,  
- Siap digunakan untuk analisis penjualan dan prediksi stok obat di Apotek XYZ.  

---

> 🟢 **Kesimpulan Akhir:**  
> Dengan penerapan metode imputasi yang tepat (**Median & Mode**), dataset menjadi bersih dan layak digunakan untuk tahap analisis selanjutnya tanpa mengorbankan keakuratan nilai penjualan maupun struktur data.